# Huber Regression Artwork Pricer

Robust log-linear Huber regression baseline for artwork valuation.

This notebook mirrors the Ridge setup but uses `HuberRegressor`, which reduces the influence of extreme residuals during training.

It trains two comparable models:
- `Huber Regression - No Estimates`
- `Huber Regression - With Log Estimates`

Both models train on `log1p(ah_price_usd)` and export predictions back on the USD price scale.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from huber_pricer.features import (
    load_artwork_data,
    split_train_test,
    make_model_frame,
    fit_feature_schema,
    apply_feature_schema,
    print_feature_summary,
)
from huber_pricer.modeling import (
    DEFAULT_ALPHAS,
    DEFAULT_EPSILONS,
    fit_huber_log_price_model,
    predict_price_from_log_model,
    evaluate_price_predictions,
    evaluate_by_quarter,
    evaluate_by_price_bucket,
    get_top_coefficients,
    huber_training_history,
)

DATA_DIR = Path("../../Data/sampled_sets_100k")
TARGET_COL = "ah_price_usd"
TRAIN_ROWS = 40_000
TEST_ROWS_PER_QUARTER = 1_000
SPLIT_RANDOM_STATE = 42
SPLIT_KEY_COL = "lot_url"
CAP_OVERSIZED_REQUESTS = True
MIN_CATEGORY_COUNT = 20
VERBOSE_PREPROCESSING = True
VERBOSE_MODEL = True

HUBER_ALPHAS = DEFAULT_ALPHAS
HUBER_EPSILONS = DEFAULT_EPSILONS
HUBER_VALIDATION_FRACTION = 0.10
HUBER_EVAL_METRIC = "log_RMSE"
HUBER_MAX_ITER = 300

MODEL_NO_ESTIMATES = "Huber Regression - No Estimates"
MODEL_WITH_ESTIMATES = "Huber Regression - With Log Estimates"

## 1. Load Sampled Data

In [2]:
raw_df = load_artwork_data(DATA_DIR, verbose=VERBOSE_PREPROCESSING)
print(raw_df.shape)
raw_df[["source_file", "source_quarter", "sale_date", "ah_price_usd", "lot_url"]].head()

[huber features] Loading artwork files from /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_100k
[huber features]   loaded Artwork_2026_finetune_100000_2022_2024_train_all_eligible.csv: 34,260 rows, 70 columns
[huber features]   loaded Artwork_2026_finetune_100000_q2025Q1_sample1000.csv: 1,000 rows, 70 columns
[huber features]   loaded Artwork_2026_finetune_100000_q2025Q2_sample1000.csv: 1,000 rows, 70 columns
[huber features]   loaded Artwork_2026_finetune_100000_q2025Q3_sample1000.csv: 1,000 rows, 70 columns
[huber features]   loaded Artwork_2026_finetune_100000_q2025Q4_sample1000.csv: 1,000 rows, 70 columns
[huber features] Loaded combined dataframe: 38,260 rows, 70 columns
(38260, 70)


,source_file,source_quarter,sale_date,ah_price_usd,lot_url
0,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2022-07-01,31771.00,https://www.christies.com/en/lot/lot-6380380?l...
1,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2024-11-20,63000.00,https://www.christies.com/en/lot/lot-6508480?l...
2,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2023-11-23,746.65,https://www.sothebys.com/en/buy/auction/2023/m...
3,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2022-09-23,11299.27,https://www.sothebys.com/en/buy/auction/2022/m...
4,Artwork_2026_finetune_100000_2022_2024_train_a...,None,2024-11-07,1852.50,https://hindmanauctions.com/auctions/2084-19th...


## 2. Train/Test Split

Uses `lot_url` for duplicate/leakage checks. If `CAP_OVERSIZED_REQUESTS = True`, oversized train/test requests are capped to available rows after filtering.

In [3]:
train_raw, test_raw = split_train_test(
    raw_df,
    train_rows=TRAIN_ROWS,
    test_rows_per_quarter=TEST_ROWS_PER_QUARTER,
    dedupe_key_col=SPLIT_KEY_COL,
    cap_oversized_requests=CAP_OVERSIZED_REQUESTS,
    random_state=SPLIT_RANDOM_STATE,
    verbose=VERBOSE_PREPROCESSING,
)
print("train raw:", train_raw.shape)
print("test raw:", test_raw.shape)
print("test quarters:")
print(test_raw["source_quarter"].value_counts().sort_index())
print("train/test lot_url overlap:", len(set(train_raw["lot_url"].dropna()) & set(test_raw["lot_url"].dropna())))

[huber features] Initial split: train=34,260, explicit test=4,000
[huber features] Requested 40,000 train rows, only 34,260 available; using all available train rows.
[huber features] Using train rows: 34,260
[huber features] De-duplicated test lot_url: 4,000->4,000
[huber features] Removed train/test overlapping lot_url values from test: 4,000->4,000
[huber features] Requested test rows per quarter: 1,000
[huber features] Actual test rows per quarter: {'2025Q1': 1000, '2025Q2': 1000, '2025Q3': 1000, '2025Q4': 1000}
[huber features] Final split: train=34,260, test=4,000
train raw: (34260, 70)
test raw: (4000, 70)
test quarters:
source_quarter
2025Q1    1000
2025Q2    1000
2025Q3    1000
2025Q4    1000
Name: count, dtype: int64
train/test lot_url overlap: 0


## 3. Feature Engineering: No Estimates

In [4]:
train_frame, feature_cols, categorical_cols, numeric_cols = make_model_frame(
    train_raw,
    target_col=TARGET_COL,
    include_estimates=False,
    verbose=VERBOSE_PREPROCESSING,
)
test_frame, _, _, _ = make_model_frame(
    test_raw,
    target_col=TARGET_COL,
    include_estimates=False,
    verbose=VERBOSE_PREPROCESSING,
)

schema = fit_feature_schema(
    train_frame,
    feature_cols,
    categorical_cols,
    min_category_count=MIN_CATEGORY_COUNT,
    verbose=VERBOSE_PREPROCESSING,
)

X_train = apply_feature_schema(train_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE_PREPROCESSING)
X_test = apply_feature_schema(test_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE_PREPROCESSING)
y_train_log = train_frame["log_target_price_usd"]

print("X_train no estimates:", X_train.shape)
print("X_test no estimates:", X_test.shape)
feature_report = print_feature_summary(
    train_frame,
    feature_cols,
    categorical_cols,
    numeric_cols,
    schema=schema,
)
feature_report

[huber features] Building Huber model frame from 34,260 rows using target_col='ah_price_usd', include_estimates=False
[huber features] Filtered invalid/non-positive targets: 34,260->34,260
[huber features] Engineered 24 features (10 categorical, 14 numeric).
[huber features]   auction estimate fields retained for reporting but excluded from model features
[huber features] Building Huber model frame from 4,000 rows using target_col='ah_price_usd', include_estimates=False
[huber features] Filtered invalid/non-positive targets: 4,000->4,000
[huber features] Engineered 24 features (10 categorical, 14 numeric).
[huber features]   auction estimate fields retained for reporting but excluded from model features
[huber features] Fitting Huber feature schema with min_category_count=20
[huber features]   categorical auction_house: 7 kept levels from 5 observed
[huber features]   categorical location: 13 kept levels from 12 observed
[huber features]   categorical material_clean: 121 kept levels fr

,feature,role,transformation,missing_rate,unique_count_before_grouping,train_schema_levels,example_values
0,auction_house,categorical,Cleaned categorical from raw auction_house; ra...,0.000000,5,7.0,"Christies, Sotheby's, Hindman"
1,location,categorical,Cleaned categorical from raw location; rare le...,0.000000,12,13.0,"London, New York, Hong Kong"
2,material_clean,categorical,Lexically cleaned raw material text; rare clea...,0.000000,10658,121.0,"coloured pencil graphite paper, cast stone, wa..."
3,artwork_title_clean,categorical,Cleaned categorical from artwork_title; rare t...,0.000000,29326,17.0,"""Stamped Indelibly"" mouth, Liseur dans un faut..."
4,medium,categorical,Cleaned categorical from raw medium; rare leve...,0.000000,12,12.0,"drawing, sculpture, painting"
5,signed,categorical,Cleaned categorical yes/no value from signed.,0.000000,2,4.0,"yes, no"
6,titled,categorical,Cleaned categorical yes/no value from titled.,0.000000,2,4.0,"yes, no"
7,dated,categorical,Cleaned categorical yes/no value from dated.,0.000000,2,4.0,"yes, no"
8,artist_name_grouped,categorical,Cleaned categorical from artist_name; rare art...,0.000000,9983,250.0,"Tom Wesselmann, Jacques Lipchitz, Dong Kingman"
9,edition_clean,categorical,Cleaned categorical from edition; rare levels ...,0.000000,1,2.0,__missing__


## 4. Feature Engineering: With Log Estimates

Uses cleaned/log-scaled estimate inputs:
- `log_auction_low_estimate_usd`
- `log_auction_high_estimate_usd`
- `log_auction_estimate_midpoint_usd`
- `auction_estimate_range_ratio`

Raw estimate USD values are retained for reporting/export only.

In [5]:
train_frame_est, feature_cols_est, categorical_cols_est, numeric_cols_est = make_model_frame(
    train_raw,
    target_col=TARGET_COL,
    include_estimates=True,
    verbose=VERBOSE_PREPROCESSING,
)
test_frame_est, _, _, _ = make_model_frame(
    test_raw,
    target_col=TARGET_COL,
    include_estimates=True,
    verbose=VERBOSE_PREPROCESSING,
)

schema_est = fit_feature_schema(
    train_frame_est,
    feature_cols_est,
    categorical_cols_est,
    min_category_count=MIN_CATEGORY_COUNT,
    verbose=VERBOSE_PREPROCESSING,
)

X_train_est = apply_feature_schema(train_frame_est, feature_cols_est, categorical_cols_est, schema_est, verbose=VERBOSE_PREPROCESSING)
X_test_est = apply_feature_schema(test_frame_est, feature_cols_est, categorical_cols_est, schema_est, verbose=VERBOSE_PREPROCESSING)

print("X_train with estimates:", X_train_est.shape)
print("X_test with estimates:", X_test_est.shape)
feature_report_est = print_feature_summary(
    train_frame_est,
    feature_cols_est,
    categorical_cols_est,
    numeric_cols_est,
    schema=schema_est,
)
feature_report_est

[huber features] Building Huber model frame from 34,260 rows using target_col='ah_price_usd', include_estimates=True
[huber features] Filtered invalid/non-positive targets: 34,260->34,260
[huber features] Engineered 28 features (10 categorical, 18 numeric).
[huber features]   included auction estimate numeric features: ['log_auction_low_estimate_usd', 'log_auction_high_estimate_usd', 'log_auction_estimate_midpoint_usd', 'auction_estimate_range_ratio']
[huber features] Building Huber model frame from 4,000 rows using target_col='ah_price_usd', include_estimates=True
[huber features] Filtered invalid/non-positive targets: 4,000->4,000
[huber features] Engineered 28 features (10 categorical, 18 numeric).
[huber features]   included auction estimate numeric features: ['log_auction_low_estimate_usd', 'log_auction_high_estimate_usd', 'log_auction_estimate_midpoint_usd', 'auction_estimate_range_ratio']
[huber features] Fitting Huber feature schema with min_category_count=20
[huber features]  

,feature,role,transformation,missing_rate,unique_count_before_grouping,train_schema_levels,example_values
0,auction_house,categorical,Cleaned categorical from raw auction_house; ra...,0.000000,5,7.0,"Christies, Sotheby's, Hindman"
1,location,categorical,Cleaned categorical from raw location; rare le...,0.000000,12,13.0,"London, New York, Hong Kong"
2,material_clean,categorical,Lexically cleaned raw material text; rare clea...,0.000000,10658,121.0,"coloured pencil graphite paper, cast stone, wa..."
3,artwork_title_clean,categorical,Cleaned categorical from artwork_title; rare t...,0.000000,29326,17.0,"""Stamped Indelibly"" mouth, Liseur dans un faut..."
4,medium,categorical,Cleaned categorical from raw medium; rare leve...,0.000000,12,12.0,"drawing, sculpture, painting"
5,signed,categorical,Cleaned categorical yes/no value from signed.,0.000000,2,4.0,"yes, no"
6,titled,categorical,Cleaned categorical yes/no value from titled.,0.000000,2,4.0,"yes, no"
7,dated,categorical,Cleaned categorical yes/no value from dated.,0.000000,2,4.0,"yes, no"
8,artist_name_grouped,categorical,Cleaned categorical from artist_name; rare art...,0.000000,9983,250.0,"Tom Wesselmann, Jacques Lipchitz, Dong Kingman"
9,edition_clean,categorical,Cleaned categorical from edition; rare levels ...,0.000000,1,2.0,__missing__


## 5. Train Huber Models With Diagnostics

In [6]:
huber_model = fit_huber_log_price_model(
    X_train,
    y_train_log,
    categorical_cols=categorical_cols,
    numeric_cols=numeric_cols,
    alphas=HUBER_ALPHAS,
    epsilons=HUBER_EPSILONS,
    validation_fraction=HUBER_VALIDATION_FRACTION,
    eval_metric=HUBER_EVAL_METRIC,
    random_state=SPLIT_RANDOM_STATE,
    max_iter=HUBER_MAX_ITER,
    verbose=VERBOSE_MODEL,
)
test_pred = predict_price_from_log_model(huber_model, X_test)

huber_model_est = fit_huber_log_price_model(
    X_train_est,
    y_train_log,
    categorical_cols=categorical_cols_est,
    numeric_cols=numeric_cols_est,
    alphas=HUBER_ALPHAS,
    epsilons=HUBER_EPSILONS,
    validation_fraction=HUBER_VALIDATION_FRACTION,
    eval_metric=HUBER_EVAL_METRIC,
    random_state=SPLIT_RANDOM_STATE,
    max_iter=HUBER_MAX_ITER,
    verbose=VERBOSE_MODEL,
)
test_pred_est = predict_price_from_log_model(huber_model_est, X_test_est)

print("Selected no-estimates alpha/epsilon:", huber_model.selected_alpha_, huber_model.selected_epsilon_)
print("Selected with-estimates alpha/epsilon:", huber_model_est.selected_alpha_, huber_model_est.selected_epsilon_)
print("Final no-estimates outlier fraction:", huber_model.outlier_fraction_)
print("Final with-estimates outlier fraction:", huber_model_est.outlier_fraction_)

training_history = huber_training_history(huber_model)
training_history.insert(0, "model", MODEL_NO_ESTIMATES)

training_history_est = huber_training_history(huber_model_est)
training_history_est.insert(0, "model", MODEL_WITH_ESTIMATES)

training_history_comparison = pd.concat([training_history, training_history_est], ignore_index=True)
training_history_comparison

[huber model] Training Huber on 34,260 rows and 24 features
[huber model] Alpha grid: [0.0001, 0.001, 0.01]
[huber model] Epsilon grid: [1.35, 1.75]
[huber model] Running Huber diagnostics: train_fit=30,834, validation=3,426, candidates=6, eval_metric=log_RMSE


/home/vann/.local/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[huber model] [01/06] alpha=0.0001 epsilon=1.35 log_RMSE=1.48598 log_MAE=1.13485 RMSE=1234028.38 MAE=197359.07 outlier_fraction=0.367


/home/vann/.local/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[huber model] [02/06] alpha=0.0001 epsilon=1.75 log_RMSE=1.48513 log_MAE=1.13719 RMSE=1236830.13 MAE=198700.51 outlier_fraction=0.133


...

[huber model] [05/06] alpha=0.01 epsilon=1.35 log_RMSE=0.61004 log_MAE=0.44474 RMSE=410131.61 MAE=68496.29 outlier_fraction=0.365


/home/vann/.local/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[huber model] [06/06] alpha=0.01 epsilon=1.75 log_RMSE=0.60873 log_MAE=0.44629 RMSE=420615.40 MAE=70751.27 outlier_fraction=0.146
[huber model] Selected alpha=0.001, epsilon=1.75
[huber model] Final fit: alpha=0.001, epsilon=1.75, n_iter=300, outlier_fraction=0.145
Selected no-estimates alpha/epsilon: 0.0001 1.75
Selected with-estimates alpha/epsilon: 0.001 1.75
Final no-estimates outlier fraction: 0.13423817863397547
Final with-estimates outlier fraction: 0.14471687098657326


/home/vann/.local/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


,model,iteration,alpha,epsilon,log_MAE,log_RMSE,log_R2,MAE,RMSE,MAPE,R2,n_iter,outlier_fraction
0,Huber Regression - No Estimates,1,0.0001,1.35,1.134849,1.485985,0.512831,197359.069986,1.234028e+06,2.639517,0.107548,300,0.367322
1,Huber Regression - No Estimates,2,0.0001,1.75,1.137187,1.485126,0.513394,198700.511514,1.236830e+06,2.642802,0.103491,300,0.133230
2,Huber Regression - No Estimates,3,0.0010,1.35,1.134788,1.485787,0.512960,197429.924340,1.234441e+06,2.638252,0.106951,300,0.367322
3,Huber Regression - No Estimates,4,0.0010,1.75,1.137352,1.485216,0.513335,198730.022794,1.236697e+06,2.641837,0.103684,300,0.133132
4,Huber Regression - No Estimates,5,0.0100,1.35,1.134679,1.485819,0.512939,197412.696505,1.234138e+06,2.638169,0.107390,300,0.366803
5,Huber Regression - No Estimates,6,0.0100,1.75,1.137210,1.485148,0.513379,198671.121001,1.236512e+06,2.641378,0.103951,300,0.133359
6,Huber Regression - With Log Estimates,1,0.0001,1.35,0.444704,0.610030,0.917898,68479.159674,4.102323e+05,0.494744,0.901373,300,0.365149
7,Huber Regression - With Log Estimates,2,0.0001,1.75,0.446231,0.608684,0.918260,70713.273636,4.203569e+05,0.505797,0.896445,300,0.146105
8,Huber Regression - With Log Estimates,3,0.0010,1.35,0.444655,0.609984,0.917910,68435.312476,4.095670e+05,0.494574,0.901693,300,0.365700
9,Huber Regression - With Log Estimates,4,0.0010,1.75,0.446246,0.608665,0.918265,70791.724609,4.209391e+05,0.505900,0.896158,300,0.146267


## 6. Overall Metrics

In [7]:
overall_metrics = pd.DataFrame([
    evaluate_price_predictions(test_frame["target_price_usd"], test_pred, label=MODEL_NO_ESTIMATES),
    evaluate_price_predictions(test_frame_est["target_price_usd"], test_pred_est, label=MODEL_WITH_ESTIMATES),
])
overall_metrics

,model,n,MAE,RMSE,MAPE,R2
0,Huber Regression - No Estimates,4000,117937.216675,789047.719788,2.710860,-0.239032
1,Huber Regression - With Log Estimates,4000,38204.999513,223528.440600,0.727329,0.900564


## 7. Quarterly Metrics

In [8]:
quarter_metrics = pd.concat(
    [
        evaluate_by_quarter(test_frame, test_pred, label=MODEL_NO_ESTIMATES),
        evaluate_by_quarter(test_frame_est, test_pred_est, label=MODEL_WITH_ESTIMATES),
    ],
    ignore_index=True,
).sort_values(["quarter", "model"]).reset_index(drop=True)
quarter_metrics

,model,n,MAE,RMSE,MAPE,R2,quarter
0,Huber Regression - No Estimates,1000,67992.165702,3.833227e+05,2.104181,0.134579,2025Q1
1,Huber Regression - With Log Estimates,1000,28170.746965,1.825941e+05,0.673327,0.803632,2025Q1
2,Huber Regression - No Estimates,1000,200476.286178,1.217828e+06,2.098565,-0.492380,2025Q2
3,Huber Regression - With Log Estimates,1000,49380.499818,2.070550e+05,0.838186,0.956860,2025Q2
4,Huber Regression - No Estimates,1000,38080.499530,2.518006e+05,4.057414,-0.953543,2025Q3
5,Huber Regression - With Log Estimates,1000,13181.798002,6.121590e+04,0.814282,0.884538,2025Q3
6,Huber Regression - No Estimates,1000,165199.915293,8.927152e+05,2.583279,-0.000311,2025Q4
7,Huber Regression - With Log Estimates,1000,62086.953266,3.462659e+05,0.583520,0.849503,2025Q4


## 8. Metrics By Price Range

In [9]:
price_bucket_metrics = pd.concat(
    [
        evaluate_by_price_bucket(test_frame, test_pred, label=MODEL_NO_ESTIMATES),
        evaluate_by_price_bucket(test_frame_est, test_pred_est, label=MODEL_WITH_ESTIMATES),
    ],
    ignore_index=True,
).sort_values(["price_bucket", "model"]).reset_index(drop=True)
price_bucket_metrics

,model,n,MAE,RMSE,MAPE,R2,price_bucket
0,Huber Regression - No Estimates,302,2.442196e+05,1.297770e+06,0.953707,-149.329873,$100k-$500k
1,Huber Regression - With Log Estimates,302,7.867628e+04,1.048073e+05,0.372336,0.019534,$100k-$500k
2,Huber Regression - No Estimates,1256,6.649097e+03,1.738777e+04,3.015280,-240.823356,$1k-$5k
3,Huber Regression - With Log Estimates,1256,1.895000e+03,8.324905e+03,0.784319,-54.433081,$1k-$5k
4,Huber Regression - No Estimates,633,4.875554e+04,2.482839e+05,1.026415,-151.541648,$25k-$100k
5,Huber Regression - With Log Estimates,633,1.840400e+04,2.472535e+04,0.383859,-0.512781,$25k-$100k
6,Huber Regression - No Estimates,163,2.092868e+06,3.451290e+06,0.863362,-0.612625,$500k+
7,Huber Regression - With Log Estimates,163,6.594629e+05,1.096042e+06,0.321587,0.837361,$500k+
8,Huber Regression - No Estimates,1252,1.249987e+04,2.330914e+04,1.164512,-17.966333,$5k-$25k
9,Huber Regression - With Log Estimates,1252,5.743414e+03,1.423487e+04,0.522866,-6.073554,$5k-$25k


## 9. Coefficient Inspection

In [10]:
top_coefficients = get_top_coefficients(huber_model, top_n=40)
top_coefficients.insert(0, "model", MODEL_NO_ESTIMATES)

top_coefficients_est = get_top_coefficients(huber_model_est, top_n=40)
top_coefficients_est.insert(0, "model", MODEL_WITH_ESTIMATES)

coefficient_comparison = pd.concat([top_coefficients, top_coefficients_est], ignore_index=True)
coefficient_comparison

,model,feature,coefficient,abs_coefficient
0,Huber Regression - No Estimates,categorical__artist_name_grouped_Frederick Conway,-3.648502,3.648502
1,Huber Regression - No Estimates,categorical__artist_name_grouped_Claude Monet,3.571485,3.571485
2,Huber Regression - No Estimates,categorical__artist_name_grouped_Jean-michel B...,3.312193,3.312193
3,Huber Regression - No Estimates,categorical__artist_name_grouped_Artist Unknown,-3.300372,3.300372
4,Huber Regression - No Estimates,categorical__artist_name_grouped_Jean-Michel B...,3.161404,3.161404
...,...,...,...,...
75,Huber Regression - With Log Estimates,categorical__medium_unknown,0.353984,0.353984
76,Huber Regression - With Log Estimates,categorical__artist_name_grouped_HENRI MATISSE,0.348343,0.348343
77,Huber Regression - With Log Estimates,categorical__artist_name_grouped_Lucy Bull,0.345445,0.345445
78,Huber Regression - With Log Estimates,categorical__artist_name_grouped_Caroline Walker,0.334888,0.334888


## 10. Export Results

In [11]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

scored_test = test_frame[[
    "ID",
    "lot_url",
    "source_quarter",
    "sale_date",
    "target_price_usd",
    "auction_low_estimate_usd",
    "auction_high_estimate_usd",
    "auction_estimate_midpoint_usd",
    "auction_estimate_range_usd",
    "log_auction_low_estimate_usd",
    "log_auction_high_estimate_usd",
    "log_auction_estimate_midpoint_usd",
    "auction_estimate_range_ratio",
]].copy()
scored_test["huber_pred_no_estimates_usd"] = test_pred
scored_test["huber_error_no_estimates_usd"] = scored_test["huber_pred_no_estimates_usd"] - scored_test["target_price_usd"]
scored_test["huber_pred_with_estimates_usd"] = test_pred_est
scored_test["huber_error_with_estimates_usd"] = scored_test["huber_pred_with_estimates_usd"] - scored_test["target_price_usd"]

scored_test.to_csv(output_dir / "huber_test_predictions.csv", index=False)
overall_metrics.to_csv(output_dir / "huber_overall_metrics.csv", index=False)
quarter_metrics.to_csv(output_dir / "huber_quarter_metrics.csv", index=False)
price_bucket_metrics.to_csv(output_dir / "huber_price_bucket_metrics.csv", index=False)
coefficient_comparison.to_csv(output_dir / "huber_top_coefficients.csv", index=False)
training_history_comparison.to_csv(output_dir / "huber_training_history.csv", index=False)
feature_report.to_csv(output_dir / "huber_feature_report_no_estimates.csv", index=False)
feature_report_est.to_csv(output_dir / "huber_feature_report_with_estimates.csv", index=False)

print("Wrote outputs to", output_dir.resolve())

Wrote outputs to /srv/notebooks/[6.0] ML Pricers/Linear_Regression/Huber_Regression/outputs
